# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook guides you through loading and exploring a dataset using the `mlcroissant` library, referencing dataset entities by their `@id` fields following the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Not a Python dict: no subscripting, use attributes
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll enumerate all record sets in the dataset, their fields and columns, showing their `@id`s for reference. These `@id` values will be used for data extraction and processing.

In [ ]:
# List all record sets and their fields by `@id`
record_sets = list(dataset.record_sets)
print(f"Record sets found ({len(record_sets)}):")
for rset in record_sets:
    print(f"  Record set: @id = {rset['@id']}, name = {rset.get('name', '<no name>')}")
    # List fields for this record set
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"    Fields ({len(fields)}):")
    for field in fields:
        if isinstance(field, str):
            print(f"      - field @id = {field}")
        elif isinstance(field, dict):
            print(f"      - field @id = {field.get('@id')}, name = {field.get('name', '<no name>')}")
    # List columns
    cols = rset.get('column', [])
    if isinstance(cols, dict):
        cols = [cols]
    print(f"    Columns ({len(cols)}):")
    for col in cols:
        if isinstance(col, str):
            print(f"      - column @id = {col}")
        elif isinstance(col, dict):
            print(f"      - column @id = {col.get('@id')}, name = {col.get('name', '<no name>')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. 
Use record set, field, or column `@id`s from the above overview to reference entities.

Below, data from each record set is loaded into a DataFrame and its columns are displayed.

In [ ]:
# List of all record set @id fields
record_set_ids = [rset['@id'] for rset in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id='{record_set_id}' ...")
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = df
    print(f"Columns: {list(df.columns)}\nSample rows:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Now, let's perform EDA on one of the main survey record sets. 
- We will filter for one of the numeric fields, e.g. GAD-7 score, using its column `@id` (replace as appropriate based on the actual list above).
- We'll normalize this score and group by a demographic field, e.g. education level. 
- **Note:** Update `main_record_set_id`, `gad7_id`, `education_id` below with actual `@id` values from your output above.

In [ ]:
# --- Choose record set and field ids as discovered in the overview ---
# Replace below as needed based on printed ids from previous cells

# Example record set (FIX if needed for your schema!):
main_record_set_id = record_set_ids[0]  # You may wish to pick a different record set
# Example numeric field (GAD-7 score, must be exact column name or @id from Data Overview section)
# List columns of DataFrame for confirmation
print(f"Main record set: {main_record_set_id}")
print(f"Available columns: {dataframes[main_record_set_id].columns.tolist()}")

# Replace below with actual column labels for field @id, e.g., 'gad7_score' or the '@id' string
numeric_field_id = 'gad7_score'  # Example, update if different
group_field_id = 'level_of_education'  # Example, update as needed

df = dataframes[main_record_set_id]

if numeric_field_id in df.columns and group_field_id in df.columns:
    # Filter
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[numeric_field_id + '_normalized'] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by a categorical field
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f'mean_{numeric_field_id}')
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped)
else:
    print(f"One of the fields {numeric_field_id} or {group_field_id} not found in columns. Please check available columns above.")

## 5. Visualization
Visualize the distribution of the GAD-7 scores (or another numeric field in your data) and the relationship with education level (or any other categorical field).

Replace `numeric_field_id` and `group_field_id` with exact values if needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    # Distribution plot
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna().astype(float), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

if numeric_field_id in df.columns and group_field_id in df.columns:
    # Boxplot by group
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, you used `mlcroissant` to load the FAIR² Mental Health Survey Dataset via its Croissant schema.

- Dataset metadata, available record sets, and fields were reviewed using their `@id`s.
- Data from each record set was loaded to a DataFrame for exploration.
- Common EDA steps, including filtering on a GAD-7 score, normalization, and groupwise aggregation by education level, were demonstrated.
- Distribution and groupwise plots were generated to visualize data patterns.

This approach provides a solid starting point for reproducible and schema-driven exploration of FAIR data using Python.

*To continue*: Explore other fields (e.g., PHQ-9, PSQ scores), examine missing data, and try additional groupings or advanced ML analysis referencing all dataset fields by their `@id`'s for reproducibility.